## ****DATA PREPARATION (PERSIAPAN DATA)****

---

Tahap ini menindaklanjuti seluruh temuan Data Understanding menjadi langkah konkret: pembagian data, imputasi, rekayasa fitur, dan encoding. Setiap keputusan di bawah merujuk pada temuan spesifik yang sudah diverifikasi sebelumnya, bukan asumsi baru.

### ****PEMUATAN DAN PEMBAGIAN DATA BERBASIS WAKTU****

Memuat ulang data mentah, mengonversi timestamp ke datetime, lalu membagi train menjadi set latih (Juli sampai Oktober 2025) dan set validasi (November 2025). November dipilih sebagai validasi karena Data Understanding menemukan rezim suhu, cuaca, dan harga bensinnya identik dengan periode test sampai level kota, sehingga skor validasi mencerminkan performa pada test dengan lebih akurat dibanding random split. Seluruh statistik pada tahap berikutnya dihitung hanya dari set latih untuk mencegah kebocoran informasi ke validasi.

In [1]:
import pandas as pd
import numpy as np

In [2]:
RAW_DIR = "../Dataset/Raw"
CLEAN_DIR = "../Dataset/Clean"
TARGET_COL = "utilization_rate"
TIMESTAMP_COL = "timestamp"

df_train_raw = pd.read_csv(f"{RAW_DIR}/train.csv", parse_dates=[TIMESTAMP_COL])
df_test = pd.read_csv(f"{RAW_DIR}/test.csv", parse_dates=[TIMESTAMP_COL])

print(f"Train mentah    : {df_train_raw.shape[0]:,} baris, {df_train_raw.shape[1]} kolom")
print(f"Test            : {df_test.shape[0]:,} baris, {df_test.shape[1]} kolom")
print(f"Rentang train   : {df_train_raw[TIMESTAMP_COL].min()} s/d {df_train_raw[TIMESTAMP_COL].max()}")
print(f"Rentang test    : {df_test[TIMESTAMP_COL].min()} s/d {df_test[TIMESTAMP_COL].max()}")

Train mentah    : 1,054,200 baris, 21 kolom
Test            : 263,550 baris, 20 kolom
Rentang train   : 2025-07-01 00:00:00 s/d 2025-11-24 09:30:00
Rentang test    : 2025-11-24 10:00:00 s/d 2025-12-31 00:00:00


In [3]:
VALIDATION_START = "2025-11-01"

is_validation = df_train_raw[TIMESTAMP_COL] >= VALIDATION_START
df_fit = df_train_raw.loc[~is_validation].reset_index(drop=True)
df_val = df_train_raw.loc[is_validation].reset_index(drop=True)

print(f"Set latih    : {df_fit.shape[0]:,} baris | {df_fit[TIMESTAMP_COL].min()} s/d {df_fit[TIMESTAMP_COL].max()}")
print(f"Set validasi : {df_val.shape[0]:,} baris | {df_val[TIMESTAMP_COL].min()} s/d {df_val[TIMESTAMP_COL].max()}")
print(f"Proporsi validasi: {len(df_val) / len(df_train_raw) * 100:.1f}%")

assert df_fit[TIMESTAMP_COL].max() < df_val[TIMESTAMP_COL].min(), "Ditemukan tumpang tindih waktu antara latih dan validasi"
print("Tidak ada tumpang tindih waktu antara set latih dan validasi.")

Set latih    : 885,600 baris | 2025-07-01 00:00:00 s/d 2025-10-31 23:30:00
Set validasi : 168,600 baris | 2025-11-01 00:00:00 s/d 2025-11-24 09:30:00
Proporsi validasi: 16.0%
Tidak ada tumpang tindih waktu antara set latih dan validasi.


### ****IMPUTASI NILAI KOSONG BERBASIS WAKTU****

Data Understanding menemukan nilai kosong pada temperature_f (sekitar 4,2%) dan precipitation_mm (sekitar 2,3%) bersifat acak sepenuhnya (MCAR) dengan blok kosong berurutan yang sangat pendek (median 1 baris, maksimum 4 baris/2 jam), sehingga interpolasi berbasis tetangga waktu adalah metode yang tepat, bukan rata-rata global.

Mengisi nilai kosong pada temperature_f dan precipitation_mm dengan interpolasi linear per stasiun, dalam blok waktu yang weather_condition-nya sama secara berurutan. Pengelompokan tambahan ini mencegah interpolasi melompati batas transisi cuaca (misalnya dari freezing ke cloudy), yang dapat menghasilkan suhu di luar rentang wajar kategori cuacanya — sejalan dengan temuan Data Understanding bahwa nilai ekstrem cuaca adalah kejadian nyata, bukan noise sensor, sehingga validitasnya harus dijaga. Interpolasi dilakukan terpisah pada set latih, validasi, dan test agar tidak ada informasi yang bocor antarset.

In [4]:
MISSING_COLS = ["temperature_f", "precipitation_mm"]


def impute_by_time_interpolation(df: pd.DataFrame, cols: list, station_col: str = "station_id",
                                  condition_col: str = "weather_condition",
                                  time_col: str = TIMESTAMP_COL) -> pd.DataFrame:
    """Isi nilai kosong dengan interpolasi linear waktu per stasiun, dalam blok weather_condition yang sama.

    Segmen dibentuk setiap weather_condition berganti pada stasiun yang sama, agar
    interpolasi tidak mencampur nilai lintas transisi cuaca. limit_direction="both"
    menangani nilai kosong di ujung segmen dengan nilai terdekat yang tersedia dalam segmen itu.
    """
    df_sorted = df.sort_values([station_col, time_col]).copy()
    condition_changed = df_sorted[condition_col] != df_sorted.groupby(station_col)[condition_col].shift()
    segment_id = condition_changed.groupby(df_sorted[station_col]).cumsum()

    df_sorted[cols] = df_sorted.groupby([station_col, segment_id])[cols].transform(
        lambda s: s.interpolate(method="linear", limit_direction="both")
    )
    return df_sorted.sort_index()


is_validation = df_train_raw[TIMESTAMP_COL] >= VALIDATION_START
df_fit = df_train_raw.loc[~is_validation].reset_index(drop=True)
df_val = df_train_raw.loc[is_validation].reset_index(drop=True)
df_test = pd.read_csv(f"{RAW_DIR}/test.csv", parse_dates=[TIMESTAMP_COL])

missing_before = {name: df[MISSING_COLS].isna().sum().sum()
                  for name, df in [("fit", df_fit), ("val", df_val), ("test", df_test)]}

df_fit = impute_by_time_interpolation(df_fit, MISSING_COLS)
df_val = impute_by_time_interpolation(df_val, MISSING_COLS)
df_test = impute_by_time_interpolation(df_test, MISSING_COLS)

for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    n_after = df_part[MISSING_COLS].isna().sum().sum()
    print(f"{name}: kosong sebelum {missing_before[name]:,} -> sesudah {n_after:,}")

fit: kosong sebelum 57,233 -> sesudah 29,457
val: kosong sebelum 10,959 -> sesudah 6,270
test: kosong sebelum 17,106 -> sesudah 9,680


In [5]:
for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    print(f"Rentang temperature_f per weather_condition ({name}), setelah imputasi:")
    display(df_part.groupby("weather_condition")["temperature_f"].agg(["mean", "min", "max"]).round(2))

Rentang temperature_f per weather_condition (fit), setelah imputasi:


,mean,min,max
weather_condition,,,
clear,78.33,45.0,95.0
cloudy,78.36,45.1,95.0
extreme_heat,100.45,95.1,110.0
heavy_rain,81.12,45.2,109.5
light_rain,81.27,45.1,109.8
partly_cloudy,78.33,45.2,95.0


Rentang temperature_f per weather_condition (val), setelah imputasi:


,mean,min,max
weather_condition,,,
clear,55.05,32.0,79.9
cloudy,55.10,32.0,79.9
freezing,27.87,20.1,31.9
heavy_rain,45.91,20.3,79.5
light_rain,45.91,20.1,79.8
partly_cloudy,55.08,32.0,79.8


Rentang temperature_f per weather_condition (test), setelah imputasi:


,mean,min,max
weather_condition,,,
clear,55.14,32.0,79.9
cloudy,55.22,32.0,79.9
freezing,27.89,20.1,31.9
heavy_rain,46.04,20.3,79.1
light_rain,45.72,20.2,79.8
partly_cloudy,55.10,32.0,79.9


#### Temuan: Imputasi Tahap 1

Interpolasi berhasil mengisi kurang dari separuh nilai kosong: fit turun dari 57.233 menjadi 29.457 (48,5%), val dari 10.959 menjadi 6.270 (42,8%), test dari 17.106 menjadi 9.680 (43,4%). Sisanya adalah titik yang berada pada segmen weather_condition tunggal (tidak punya tetangga sejenis untuk diinterpolasi). Rentang suhu tiap kategori cuaca tetap valid (misalnya extreme_heat 95,1–110,0°F, sesuai batas asli pada Data Understanding), mengonfirmasi metode segmentasi per weather_condition bekerja. Tahap kedua diperlukan untuk mengisi sisanya.

### ****IMPUTASI SISA NILAI KOSONG (RATA-RATA KONDISIONAL)****

Mengisi sisa nilai kosong yang tidak berhasil diinterpolasi karena berada pada segmen tunggal (weather_condition hanya muncul satu baris pada blok waktu tersebut). Menggunakan rata-rata (station_id, weather_condition), dengan fallback ke rata-rata weather_condition saja bila kombinasi tersebut tidak tersedia. Statistik dihitung dari partisi yang sama (val dari val, test dari test), bukan hanya dari set latih, karena temperature_f dan precipitation_mm adalah fitur eksogen yang sudah diketahui nilainya untuk seluruh baris test (bukan target), sehingga tidak ada risiko kebocoran informasi target. Ini juga diperlukan karena weather_condition freezing sama sekali tidak muncul pada set latih (Juli sampai Oktober).

In [8]:
def fill_missing_with_conditional_mean(df: pd.DataFrame, cols: list,
                                        primary_group: list, fallback_group: list) -> pd.DataFrame:
    """Isi sisa nilai kosong dengan rata-rata kondisional dari partisi yang sama.

    Fallback dua tingkat: rata-rata primary_group, lalu rata-rata fallback_group
    bila kombinasi primary_group tidak tersedia (mean() mengabaikan NaN secara default).
    """
    df = df.copy()
    df[cols] = df[cols].fillna(df.groupby(primary_group)[cols].transform("mean"))
    df[cols] = df[cols].fillna(df.groupby(fallback_group)[cols].transform("mean"))
    return df


PRIMARY_GROUP = ["station_id", "weather_condition"]
FALLBACK_GROUP = ["weather_condition"]

missing_before_tier2 = {name: df[MISSING_COLS].isna().sum().sum()
                        for name, df in [("fit", df_fit), ("val", df_val), ("test", df_test)]}

df_fit = fill_missing_with_conditional_mean(df_fit, MISSING_COLS, PRIMARY_GROUP, FALLBACK_GROUP)
df_val = fill_missing_with_conditional_mean(df_val, MISSING_COLS, PRIMARY_GROUP, FALLBACK_GROUP)
df_test = fill_missing_with_conditional_mean(df_test, MISSING_COLS, PRIMARY_GROUP, FALLBACK_GROUP)

for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    n_after = df_part[MISSING_COLS].isna().sum().sum()
    print(f"{name}: kosong sebelum tahap 2 {missing_before_tier2[name]:,} -> sesudah {n_after:,}")

fit: kosong sebelum tahap 2 0 -> sesudah 0
val: kosong sebelum tahap 2 0 -> sesudah 0
test: kosong sebelum tahap 2 0 -> sesudah 0


In [9]:
for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    print(f"Rentang temperature_f per weather_condition ({name}), setelah imputasi lengkap:")
    display(df_part.groupby("weather_condition")["temperature_f"].agg(["mean", "min", "max"]).round(2))

Rentang temperature_f per weather_condition (fit), setelah imputasi lengkap:


,mean,min,max
weather_condition,,,
clear,78.35,45.0,95.0
cloudy,78.37,45.1,95.0
extreme_heat,100.45,95.1,110.0
heavy_rain,81.14,45.2,109.5
light_rain,81.27,45.1,109.8
partly_cloudy,78.34,45.2,95.0


Rentang temperature_f per weather_condition (val), setelah imputasi lengkap:


,mean,min,max
weather_condition,,,
clear,54.99,32.0,79.9
cloudy,55.04,32.0,79.9
freezing,27.87,20.1,31.9
heavy_rain,45.81,20.3,79.5
light_rain,45.96,20.1,79.8
partly_cloudy,55.05,32.0,79.8


Rentang temperature_f per weather_condition (test), setelah imputasi lengkap:


,mean,min,max
weather_condition,,,
clear,55.09,32.0,79.9
cloudy,55.17,32.0,79.9
freezing,27.89,20.1,31.9
heavy_rain,46.10,20.3,79.1
light_rain,45.74,20.2,79.8
partly_cloudy,55.05,32.0,79.9


#### Temuan: Imputasi Tahap 2 (Lengkap)

Seluruh nilai kosong terisi (0 sisa di fit, val, dan test). Rentang tiap kategori cuaca tetap konsisten dengan batas aslinya setelah imputasi lengkap: freezing 20,1–31,9°F pada val dan test (kategori yang sama sekali tidak ada di fit), extreme_heat 95,1–110,0°F pada fit. Ini mengonfirmasi rata-rata kondisional (station_id, weather_condition) berhasil mengisi sisa nilai kosong tanpa mencemari validitas kategori cuaca, termasuk untuk kategori yang tidak pernah dipelajari dari fit.

### ****REKAYASA FITUR TEMPORAL****

Data Understanding menemukan pola jam yang kuat dan bimodal (rendah dini hari, dua puncak siang dan sore untuk sebagian tipe lokasi) serta perbedaan rata-rata sekitar 0,035–0,04 antara hari kerja dan akhir pekan, sehingga hour dan day_of_week wajib menjadi fitur inti.

Menurunkan fitur waktu dari timestamp: hour, day_of_week, month, dan is_weekend, ditambah encoding siklikal (sin/cos) untuk hour dan day_of_week. Encoding siklikal diperlukan karena jam 23 dan jam 0 sebenarnya berdekatan secara waktu, tetapi secara nilai numerik biasa terlihat jauh (perbedaan 23), sehingga model berbasis jarak atau linear bisa salah menafsirkannya. Fitur mentah (hour, day_of_week) tetap disertakan karena model berbasis pohon (Random Forest, LightGBM, XGBoost) dapat memanfaatkan nilai kategorikal/ordinal secara langsung tanpa perlu encoding siklikal.

In [12]:
def add_temporal_features(df: pd.DataFrame, time_col: str = TIMESTAMP_COL) -> pd.DataFrame:
    """Turunkan fitur waktu dari timestamp, termasuk encoding siklikal hour dan day_of_week."""
    df = df.copy()
    df["hour"] = df[time_col].dt.hour
    df["day_of_week"] = df[time_col].dt.dayofweek
    df["month"] = df[time_col].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    return df


df_fit = add_temporal_features(df_fit)
df_val = add_temporal_features(df_val)
df_test = add_temporal_features(df_test)

temporal_cols = ["hour", "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
display(df_fit[[TIMESTAMP_COL] + temporal_cols].head())

,timestamp,hour,day_of_week,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos
0,2025-07-01,0,1,7,0,0.0,1.0,0.781831,0.62349
1,2025-07-01,0,1,7,0,0.0,1.0,0.781831,0.62349
2,2025-07-01,0,1,7,0,0.0,1.0,0.781831,0.62349
3,2025-07-01,0,1,7,0,0.0,1.0,0.781831,0.62349
4,2025-07-01,0,1,7,0,0.0,1.0,0.781831,0.62349


In [13]:
print("Rentang nilai fitur temporal (fit):")
display(df_fit[temporal_cols].agg(["min", "max"]).T)

print(f"is_weekend berdasarkan day_of_week: {(df_fit['is_weekend'] == (df_fit['day_of_week'] >= 5).astype(int)).all()}")
print(f"hour_sin^2 + hour_cos^2 mendekati 1: {np.allclose(df_fit['hour_sin']**2 + df_fit['hour_cos']**2, 1)}")

Rentang nilai fitur temporal (fit):


,min,max
hour,0.000000,23.000000
day_of_week,0.000000,6.000000
month,7.000000,10.000000
is_weekend,0.000000,1.000000
hour_sin,-1.000000,1.000000
hour_cos,-1.000000,1.000000
dow_sin,-0.974928,0.974928
dow_cos,-0.900969,1.000000


is_weekend berdasarkan day_of_week: True
hour_sin^2 + hour_cos^2 mendekati 1: True


#### Temuan: Validasi Fitur Temporal

Seluruh fitur berada pada rentang yang benar (hour 0–23, day_of_week 0–6, month 7–10 sesuai periode fit, is_weekend 0/1), is_weekend konsisten 100% dengan day_of_week, dan identitas trigonometri hour_sin² + hour_cos² ≈ 1 terpenuhi. Fitur temporal siap dipakai tanpa anomali.

### ****FITUR HISTORIS: RATA-RATA UTILIZATION PER STASIUN DAN PER (LOCATION_TYPE, HOUR)****

Membuat dua fitur target encoding: rata-rata utilization_rate historis per station_id, dan per kombinasi (location_type, hour), berdasarkan temuan bahwa keduanya adalah pembeda kuat pada Data Understanding.

Perhitungan naif (rata-rata langsung dari baris yang sama) menyebabkan kebocoran, karena nilai target baris itu sendiri ikut membentuk rata-rata yang dipakai sebagai fiturnya. Untuk set latih, dipakai leave-one-out (LOO): rata-rata grup dihitung tanpa menyertakan baris itu sendiri. Untuk set validasi dan test, dipakai rata-rata penuh dari set latih (aman karena baris val/test tidak pernah menjadi bagian dari grup yang dihitung). Kombinasi yang tidak ditemukan pada set latih (semestinya tidak terjadi karena station_id dan location_type sama persis di ketiga set) memakai rata-rata global set latih sebagai fallback.

In [14]:
def compute_loo_target_mean(df: pd.DataFrame, group_cols: list, target_col: str) -> pd.Series:
    """Rata-rata target per grup tanpa menyertakan baris itu sendiri (leave-one-out)."""
    group_sum = df.groupby(group_cols)[target_col].transform("sum")
    group_count = df.groupby(group_cols)[target_col].transform("count")
    return (group_sum - df[target_col]) / (group_count - 1)

def apply_target_mean_mapping(df: pd.DataFrame, group_mean: pd.Series, group_cols: list,
                              global_mean: float, feature_name: str) -> pd.DataFrame:
    """Terapkan rata-rata target hasil fit ke df lain (val/test) lewat merge, dengan fallback rata-rata global."""
    mapping_df = group_mean.rename(feature_name).reset_index()
    df = df.merge(mapping_df, on=group_cols, how="left")
    df[feature_name] = df[feature_name].fillna(global_mean)
    return df

GLOBAL_MEAN_FIT = df_fit[TARGET_COL].mean()
print(f"Rata-rata global utilization_rate (fit): {GLOBAL_MEAN_FIT:.4f}")

Rata-rata global utilization_rate (fit): 0.4003


In [15]:
STATION_GROUP = ["station_id"]

df_fit["station_target_mean"] = compute_loo_target_mean(df_fit, STATION_GROUP, TARGET_COL)
station_mean_mapping = df_fit.groupby(STATION_GROUP)[TARGET_COL].mean()

df_val = apply_target_mean_mapping(df_val, station_mean_mapping, STATION_GROUP, GLOBAL_MEAN_FIT, "station_target_mean")
df_test = apply_target_mean_mapping(df_test, station_mean_mapping, STATION_GROUP, GLOBAL_MEAN_FIT, "station_target_mean")

print(f"NaN pada station_target_mean -> fit: {df_fit['station_target_mean'].isna().sum()}, "
      f"val: {df_val['station_target_mean'].isna().sum()}, test: {df_test['station_target_mean'].isna().sum()}")
display(df_fit[["station_id", TARGET_COL, "station_target_mean"]].head())

NaN pada station_target_mean -> fit: 0, val: 0, test: 0


,station_id,utilization_rate,station_target_mean
0,EV00001,0.131,0.569800
1,EV00094,0.200,0.556506
2,EV00131,0.233,0.354437
3,EV00054,0.154,0.333836
4,EV00079,0.235,0.484390


In [16]:
LOC_HOUR_GROUP = ["location_type", "hour"]

df_fit["loc_hour_target_mean"] = compute_loo_target_mean(df_fit, LOC_HOUR_GROUP, TARGET_COL)
loc_hour_mean_mapping = df_fit.groupby(LOC_HOUR_GROUP)[TARGET_COL].mean()

df_val = apply_target_mean_mapping(df_val, loc_hour_mean_mapping, LOC_HOUR_GROUP, GLOBAL_MEAN_FIT, "loc_hour_target_mean")
df_test = apply_target_mean_mapping(df_test, loc_hour_mean_mapping, LOC_HOUR_GROUP, GLOBAL_MEAN_FIT, "loc_hour_target_mean")

print(f"NaN pada loc_hour_target_mean -> fit: {df_fit['loc_hour_target_mean'].isna().sum()}, "
      f"val: {df_val['loc_hour_target_mean'].isna().sum()}, test: {df_test['loc_hour_target_mean'].isna().sum()}")

NaN pada loc_hour_target_mean -> fit: 0, val: 0, test: 0


In [17]:
encoded_cols = ["station_target_mean", "loc_hour_target_mean"]

for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    print(f"Statistik fitur target encoding ({name}):")
    display(df_part[encoded_cols].describe().round(4))

print("Rata-rata loc_hour_target_mean pukul 12 (fit), harus tinggi untuk Shopping Center, rendah untuk Residential:")
display(df_fit.loc[df_fit["hour"] == 12].groupby("location_type")["loc_hour_target_mean"].mean().sort_values().round(3))

Statistik fitur target encoding (fit):


,station_target_mean,loc_hour_target_mean
count,885600.0000,885600.0000
mean,0.4003,0.4003
std,0.0954,0.2607
min,0.2146,0.0227
25%,0.3348,0.1157
50%,0.3928,0.4285
75%,0.4709,0.6342
max,0.6364,0.8596


Statistik fitur target encoding (val):


,station_target_mean,loc_hour_target_mean
count,168600.0000,168600.0000
mean,0.4003,0.3967
std,0.0954,0.2609
min,0.2146,0.0227
25%,0.3348,0.1132
50%,0.3928,0.4236
75%,0.4709,0.6341
max,0.6363,0.8596


Statistik fitur target encoding (test):


,station_target_mean,loc_hour_target_mean
count,263550.0000,263550.0000
mean,0.4003,0.4024
std,0.0954,0.2606
min,0.2146,0.0227
25%,0.3348,0.1157
50%,0.3928,0.4401
75%,0.4709,0.6353
max,0.6363,0.8596


Rata-rata loc_hour_target_mean pukul 12 (fit), harus tinggi untuk Shopping Center, rendah untuk Residential:


location_type
Residential          0.122
Suburban             0.620
Hotel/Hospitality    0.654
Airport              0.655
Workplace            0.681
Highway Corridor     0.719
Urban Center         0.786
Shopping Center      0.826
Name: loc_hour_target_mean, dtype: float64

#### Temuan: Validasi Target Encoding

Tidak ada nilai kosong pada kedua fitur di ketiga set. loc_hour_target_mean memiliki sebaran jauh lebih lebar (std 0,26, rentang 0,02–0,86) dibanding station_target_mean (std 0,095, rentang 0,21–0,64), menegaskan interaksi lokasi x jam sebagai pembeda utama, sejalan dengan Data Understanding. Pengecekan pukul 12:00 mengonfirmasi urutan yang sama seperti Data Understanding: Shopping Center tertinggi (0,826), Residential terendah (0,122).

### ****ENCODING FITUR KATEGORIK****

Mengubah kolom kategorik (network, city, state, location_type, charger_type, pricing_type, weather_condition, local_event) menjadi kode integer (label encoding). Model berbasis pohon (Random Forest, LightGBM, XGBoost) tidak mengasumsikan urutan pada kode ini, karena setiap split memilah kategori berdasarkan nilai yang membedakan target, bukan urutan besar-kecil, sehingga label encoding lebih efisien daripada one-hot terutama untuk city yang punya 15 kategori. Mapping dibentuk hanya dari set latih dan diterapkan ke val serta test, dengan kode -1 untuk kategori yang tidak pernah muncul di set latih.

In [18]:
CATEGORICAL_COLS = ["network", "city", "state", "location_type", "charger_type",
                    "pricing_type", "weather_condition", "local_event"]


def fit_label_encoding(df: pd.DataFrame, cols: list) -> dict:
    """Bentuk mapping kategori -> kode integer dari satu dataset (set latih)."""
    return {col: {category: code for code, category in enumerate(sorted(df[col].dropna().unique()))}
            for col in cols}


def apply_label_encoding(df: pd.DataFrame, encoding_maps: dict) -> pd.DataFrame:
    """Terapkan mapping label encoding; kategori tak dikenal diberi kode -1."""
    df = df.copy()
    for col, mapping in encoding_maps.items():
        df[f"{col}_code"] = df[col].map(mapping).fillna(-1).astype(int)
    return df


label_encoding_maps = fit_label_encoding(df_fit, CATEGORICAL_COLS)

df_fit = apply_label_encoding(df_fit, label_encoding_maps)
df_val = apply_label_encoding(df_val, label_encoding_maps)
df_test = apply_label_encoding(df_test, label_encoding_maps)

for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    n_unknown = sum((df_part[f"{col}_code"] == -1).sum() for col in CATEGORICAL_COLS)
    print(f"{name}: total baris dengan kategori tak dikenal (-1) = {n_unknown}")

fit: total baris dengan kategori tak dikenal (-1) = 0
val: total baris dengan kategori tak dikenal (-1) = 23953
test: total baris dengan kategori tak dikenal (-1) = 37694


### ****FITUR JUMLAH FASILITAS SEKITAR (AMENITIES_NEARBY)****

amenities_nearby berisi teks multi-nilai dipisah koma (misalnya "Hotel, Fast Food, WiFi"). Diturunkan menjadi n_amenities, yaitu jumlah fasilitas di sekitar stasiun, sebagai proksi seberapa ramai atau strategis lokasi tersebut. Nilai mentahnya sendiri (123 kombinasi unik) tidak dipakai langsung karena kardinalitasnya terlalu tinggi untuk label encoding biasa.

In [19]:
def add_amenity_count(df: pd.DataFrame, col: str = "amenities_nearby") -> pd.DataFrame:
    """Hitung jumlah fasilitas sekitar dari teks dipisah koma."""
    df = df.copy()
    df["n_amenities"] = df[col].str.split(",").apply(len)
    return df


df_fit = add_amenity_count(df_fit)
df_val = add_amenity_count(df_val)
df_test = add_amenity_count(df_test)

display(df_fit[["amenities_nearby", "n_amenities"]].drop_duplicates().head())
print(f"Rentang n_amenities (fit): {df_fit['n_amenities'].min()} sampai {df_fit['n_amenities'].max()}")

,amenities_nearby,n_amenities
0,Hotel,1
1,"WiFi, Restroom",2
2,"Shopping Mall, Restaurant",2
3,"Park, Fast Food, WiFi",3
4,"Fast Food, Coffee Shop, Park, Convenience Stor...",5


Rentang n_amenities (fit): 1 sampai 5


In [20]:
print("Contoh mapping label encoding (location_type):")
print(label_encoding_maps["location_type"])

print("\nRata-rata utilization_rate menurut n_amenities (fit), cek apakah ada tren:")
display(df_fit.groupby("n_amenities")[TARGET_COL].agg(["mean", "count"]).round(3))

Contoh mapping label encoding (location_type):
{'Airport': 0, 'Highway Corridor': 1, 'Hotel/Hospitality': 2, 'Residential': 3, 'Shopping Center': 4, 'Suburban': 5, 'Urban Center': 6, 'Workplace': 7}

Rata-rata utilization_rate menurut n_amenities (fit), cek apakah ada tren:


,mean,count
n_amenities,,
1,0.402,165312
2,0.379,200736
3,0.408,135792
4,0.415,183024
5,0.400,200736


#### Temuan: Fitur Amenities dan Contoh Mapping

n_amenities berkisar 1–5 fasilitas per stasiun, dengan rata-rata utilization_rate hampir datar di semua nilainya (0,379–0,415). Sinyal langsungnya lemah, tetapi tetap dipertahankan karena berpotensi berguna dalam interaksi dengan fitur lain pada model berbasis pohon. Contoh mapping location_type menunjukkan 8 kode berurutan alfabetis, dipakai konsisten di ketiga set.

In [21]:
for col in CATEGORICAL_COLS:
    n_unknown_val = (df_val[f"{col}_code"] == -1).sum()
    n_unknown_test = (df_test[f"{col}_code"] == -1).sum()
    if n_unknown_val or n_unknown_test:
        print(f"{col}: val={n_unknown_val:,}, test={n_unknown_test:,}")

print(f"\nJumlah baris freezing -> val: {(df_val['weather_condition'] == 'freezing').sum():,}, "
      f"test: {(df_test['weather_condition'] == 'freezing').sum():,}")

weather_condition: val=23,953, test=37,694

Jumlah baris freezing -> val: 23,953, test: 37,694


#### Temuan: Sumber Kategori Tak Dikenal

Seluruh kode -1 (23.953 baris val, 37.694 baris test) berasal murni dari weather_condition = freezing, kategori yang sama sekali tidak muncul pada fit (Juli–Oktober), persis seperti temuan Data Understanding. Kolom kategorik lain 0 kode tak dikenal, karena 150 stasiun dan atributnya identik di ketiga set. Kode -1 tetap aman untuk model berbasis tree.

>Tetapi diingat kembali di Ringkasan: model final wajib dilatih ulang dari gabungan seluruh train agar freezing benar-benar dipelajari.

### ****PENYUSUNAN FITUR FINAL DAN PENYIMPANAN****

Menyusun daftar fitur final untuk pemodelan, mengeluarkan kolom pengenal (id, station_id, station_name, timestamp mentah), kolom kategorik mentah yang sudah diwakili oleh versi `_code`, dan amenities_nearby mentah yang sudah diwakili oleh n_amenities. Kolom id dan timestamp tetap disimpan terpisah untuk keperluan submission dan analisis lanjutan, bukan dibuang. Hasil akhir disimpan ke `Dataset/Clean` sebagai train_fit, train_val, dan test, agar tahap Modeling tidak perlu mengulang seluruh Data Preparation.

In [22]:
FEATURE_COLS = (
    ["latitude", "longitude", "power_output_kw", "ports_total", "n_amenities",
     "temperature_f", "precipitation_mm", "gas_price_per_gallon"]
    + temporal_cols
    + ["station_target_mean", "loc_hour_target_mean"]
    + [f"{col}_code" for col in CATEGORICAL_COLS]
)

ID_COLS = ["id", "station_id", TIMESTAMP_COL]

print(f"Jumlah fitur final: {len(FEATURE_COLS)}")
missing_cols = [col for col in FEATURE_COLS if col not in df_fit.columns]
print(f"Fitur hilang dari df_fit: {missing_cols}")

for name, df_part in [("fit", df_fit), ("val", df_val), ("test", df_test)]:
    n_na = df_part[FEATURE_COLS].isna().sum().sum()
    print(f"{name}: nilai kosong pada fitur final = {n_na}")

train_fit_final = pd.concat([df_fit[ID_COLS], df_fit[FEATURE_COLS], df_fit[[TARGET_COL]]], axis=1)
train_val_final = pd.concat([df_val[ID_COLS], df_val[FEATURE_COLS], df_val[[TARGET_COL]]], axis=1)
test_final = pd.concat([df_test[ID_COLS], df_test[FEATURE_COLS]], axis=1)

train_fit_final.to_csv(f"{CLEAN_DIR}/train_fit.csv", index=False)
train_val_final.to_csv(f"{CLEAN_DIR}/train_val.csv", index=False)
test_final.to_csv(f"{CLEAN_DIR}/test_clean.csv", index=False)

print(f"\ntrain_fit  : {train_fit_final.shape}")
print(f"train_val  : {train_val_final.shape}")
print(f"test_clean : {test_final.shape}")

Jumlah fitur final: 26
Fitur hilang dari df_fit: []
fit: nilai kosong pada fitur final = 0
val: nilai kosong pada fitur final = 0
test: nilai kosong pada fitur final = 0

train_fit  : (885600, 30)
train_val  : (168600, 30)
test_clean : (263550, 29)


### ****RINGKASAN DATA PREPARATION****

**Split berbasis waktu**
- train_fit: Juli sampai Oktober 2025 (885.600 baris), train_val: November 2025 (168.600 baris), dipilih karena rezim val paling mendekati periode test.

**Imputasi (temperature_f, precipitation_mm)**
- Dua tahap: interpolasi linear per stasiun dalam blok weather_condition yang sama, lalu rata-rata kondisional (station_id, weather_condition) dengan fallback rata-rata weather_condition untuk sisa blok yang tidak punya tetangga sejenis. Statistik dihitung terpisah per partisi (fit, val, test) karena keduanya fitur eksogen, bukan target. Hasil akhir 0 nilai kosong di ketiga set, dan rentang tiap kategori cuaca tetap valid.

**Fitur baru**
- Temporal: hour, day_of_week, month, is_weekend, serta encoding siklikal hour_sin, hour_cos, dow_sin, dow_cos.
- Target encoding: station_target_mean dan loc_hour_target_mean, dihitung leave-one-out pada train_fit dan dipetakan lewat merge ke val serta test agar tidak bocor.
- n_amenities: jumlah fasilitas sekitar dari amenities_nearby; sinyalnya terhadap target lemah (0,379 sampai 0,415) tapi dipertahankan untuk potensi interaksi.
- Label encoding untuk 8 kolom kategorik, mapping dibentuk dari train_fit saja.

**Catatan penting untuk Modeling**
- weather_condition = freezing sama sekali tidak ada di train_fit (Juli sampai Oktober), sehingga 23.953 baris val (14,2%) dan 37.694 baris test (14,3%) mendapat kode -1 pada weather_condition_code. Ini realistis sebagai uji generalisasi model terhadap kondisi tak terlihat, tetapi berarti skor RMSE pada val kemungkinan sedikit lebih pesimis dibanding performa asli, karena model final akan dilatih ulang dengan freezing yang sudah dikenal.
- Untuk submission final: gabungkan train_fit dan train_val menjadi satu set latih penuh, bentuk ulang seluruh mapping (label encoding, station_target_mean, loc_hour_target_mean, GLOBAL_MEAN_FIT) dari gabungan itu, baru latih model dan prediksi test_clean.
- Prediksi akhir wajib di-clip ke rentang 0,02 dan 0,98 sesuai temuan Data Understanding.
- station_id dan timestamp disimpan di file Clean untuk keperluan analisis dan penyusunan submission, dikeluarkan dari daftar FEATURE_COLS saat training.

**File tersimpan di Dataset/Clean**
- train_fit.csv (885.600 baris, 30 kolom)
- train_val.csv (168.600 baris, 30 kolom)
- test_clean.csv (263.550 baris, 29 kolom)

<div align="center">
  <img src="https://i.pinimg.com/736x/16/69/f4/1669f483958cab072d13c524e942e567.jpg" alt="Hanaori Imutt" width="300">
  <h5>Lanjut ke "Modeling" desuu </h5>
</div>